## 1. Importación de librerías

Se utilizan **Pandas** y **NumPy** para la manipulación y el análisis de datos, **Plotly** para la visualización interactiva 3D, y utilidades estándar (`json`, `pathlib`) para la construcción del panel interactivo con sprites.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, IFrame, HTML
import json
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
import plotly
print("Plotly:", plotly.__version__)


Pandas: 2.3.3
NumPy: 2.3.5
Plotly: 6.3.0


## 2. Carga del dataset

**Origen de los datos:** el archivo `Pokemon2.csv` contiene las estadísticas base de los
Pokémon de las **generaciones 1 a 6**, compiladas originalmente a partir de
[pokemondb.net](https://pokemondb.net) (metodología equivalente al repositorio público
[`lgreski/pokemonData`](https://github.com/lgreski/pokemonData)). El archivo incluye,
además, dos registros personalizados (`Sesni`, `Chuchin`) agregados como práctica del curso.

**Contenido:** 807 registros con el número de Pokédex, nombre, tipo primario y
secundario, seis estadísticas base (HP, Ataque, Defensa, Ataque especial, Defensa
especial y Velocidad), el total de estadísticas, la generación, si es legendario y su
nivel de evolución.

**Sprites:** las imágenes de cada Pokémon se obtienen del repositorio público
[`PokeAPI/sprites`](https://github.com/PokeAPI/sprites) en GitHub (carpeta
`sprites/pokemon/`), indexadas por el número de Pokédex nacional.


In [4]:
RUTA_ORIGEN = "Pokemon2.csv"
df_raw = pd.read_csv(RUTA_ORIGEN)
df_raw.head()


,dex_number,name,type_1,type_2,total,hp,attack,defense,sp_atk,sp_def,speed,generation,legendary,evolution_level,generacion_valida,promedio_estadisticas,sprite_url
0,16,Pidgey,Normal,Flying,251,40,45,40,35,35,56,1,False,1,True,41.83,https://raw.githubusercontent.com/PokeAPI/spri...
1,17,Pidgeotto,Normal,Flying,349,63,60,55,50,50,71,1,False,1,True,58.17,https://raw.githubusercontent.com/PokeAPI/spri...
2,18,Pidgeot,Normal,Flying,479,83,80,75,70,70,101,1,False,2,True,79.83,https://raw.githubusercontent.com/PokeAPI/spri...
3,18,PidgeotMega Pidgeot,Normal,Flying,579,83,80,80,135,80,121,1,False,3,True,96.50,https://raw.githubusercontent.com/PokeAPI/spri...
4,19,Rattata,Normal,Sin segundo tipo,253,30,56,35,25,35,72,1,False,1,True,42.17,https://raw.githubusercontent.com/PokeAPI/spri...


## 3. Inspección inicial del dataset

In [5]:
df_raw.head()


,dex_number,name,type_1,type_2,total,hp,attack,defense,sp_atk,sp_def,speed,generation,legendary,evolution_level,generacion_valida,promedio_estadisticas,sprite_url
0,16,Pidgey,Normal,Flying,251,40,45,40,35,35,56,1,False,1,True,41.83,https://raw.githubusercontent.com/PokeAPI/spri...
1,17,Pidgeotto,Normal,Flying,349,63,60,55,50,50,71,1,False,1,True,58.17,https://raw.githubusercontent.com/PokeAPI/spri...
2,18,Pidgeot,Normal,Flying,479,83,80,75,70,70,101,1,False,2,True,79.83,https://raw.githubusercontent.com/PokeAPI/spri...
3,18,PidgeotMega Pidgeot,Normal,Flying,579,83,80,80,135,80,121,1,False,3,True,96.50,https://raw.githubusercontent.com/PokeAPI/spri...
4,19,Rattata,Normal,Sin segundo tipo,253,30,56,35,25,35,72,1,False,1,True,42.17,https://raw.githubusercontent.com/PokeAPI/spri...


In [6]:
print("Dimensiones (filas, columnas):", df_raw.shape)


Dimensiones (filas, columnas): (803, 17)


In [7]:
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 803 entries, 0 to 802
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dex_number             803 non-null    int64  
 1   name                   803 non-null    object 
 2   type_1                 803 non-null    object 
 3   type_2                 803 non-null    object 
 4   total                  803 non-null    int64  
 5   hp                     803 non-null    int64  
 6   attack                 803 non-null    int64  
 7   defense                803 non-null    int64  
 8   sp_atk                 803 non-null    int64  
 9   sp_def                 803 non-null    int64  
 10  speed                  803 non-null    int64  
 11  generation             803 non-null    int64  
 12  legendary              803 non-null    bool   
 13  evolution_level        803 non-null    int64  
 14  generacion_valida      803 non-null    bool   
 15  promed

In [8]:
df_raw.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
dex_number,803.0,NaN,NaN,NaN,363.270237,209.099012,1.0,184.5,365.0,540.5,723.0
name,803,802,Caterpie,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type_1,803,19,Water,112,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type_2,803,19,Sin segundo tipo,387,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total,803.0,NaN,NaN,NaN,435.058531,120.791407,180.0,330.0,450.0,515.0,787.0
hp,803.0,NaN,NaN,NaN,69.291407,25.569274,1.0,50.0,65.0,80.0,255.0
attack,803.0,NaN,NaN,NaN,79.84807,38.960743,5.0,55.0,75.0,100.0,676.0
defense,803.0,NaN,NaN,NaN,74.841843,38.663987,5.0,50.0,70.0,90.0,678.0
sp_atk,803.0,NaN,NaN,NaN,73.816936,40.147734,10.0,49.5,65.0,95.0,688.0
sp_def,803.0,NaN,NaN,NaN,72.890411,36.100724,20.0,50.0,70.0,90.0,678.0
